# Day 27: Agent Memory

Yesterday we built an agent that calls APIs. But ask it a follow-up question —
it has **no idea** what you just talked about. Every call starts from scratch.

Today we give our agent **memory**.

## Install

In [ ]:
%pip install langchain langchain-google-genai langgraph requests --quiet

## Setup

In [20]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

def get_text(result):
    """Extract plain text from agent result."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return " ".join(c.get("text", "") for c in content if isinstance(c, dict))
    return content

## The Tool: Wikipedia Lookup

In [21]:
import requests
from langchain.tools import tool

HEADERS = {"User-Agent": "genai-50-days-demo/1.0"}

@tool
def wikipedia(topic: str) -> str:
    """Look up a topic on Wikipedia and return a summary."""
    slug = topic.strip().replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}"
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        return response.json().get("extract", "No summary found.")
    return f"Could not find article for: {topic}"

# Quick test
print(wikipedia.invoke("Marie Curie")[:200])

Maria Salomea Skłodowska Curie, better known as Marie Curie, was a Polish and naturalised-French physicist and chemist. She shared the 1903 Nobel Prize in Physics with her husband Pierre Curie "for th


## Add Memory: Checkpointer

In [22]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=[wikipedia],
    checkpointer=InMemorySaver()
)

print("\u2705 Agent created (with memory)")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Agent created (with memory)


## Conversation with Memory

In [23]:
config = {"configurable": {"thread_id": "chat-1"}}

result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Look up Marie Curie"}]},
    config
)
print(get_text(result))

Marie Curie was a Polish and naturalised-French physicist and chemist. She shared the 1903 Nobel Prize in Physics with her husband Pierre Curie for their joint research on radioactivity. She also won the 1911 Nobel Prize in Chemistry for the discovery of the elements radium and polonium, the isolation of radium, and the study of the nature and compounds of this remarkable element.


In [24]:
# Follow-up — same thread_id, agent remembers!
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What did she win the Nobel Prize for?"}]},
    config
)
print(get_text(result))

Marie Curie won the 1903 Nobel Prize in Physics with her husband Pierre Curie for their joint research on the radioactivity phenomena discovered by Professor Henri Becquerel. She also won the 1911 Nobel Prize in Chemistry for the discovery of the elements radium and polonium, by the isolation of radium and the study of the nature and compounds of this remarkable element.


In [ ]:
# One more follow-up
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Which country was she from?"}]},
    config
)
print(get_text(result))

## New Thread = Fresh Start

In [25]:
# Different thread_id — memory is isolated
new_config = {"configurable": {"thread_id": "chat-2"}}

result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What did she win the Nobel Prize for?"}]},
    new_config
)
print(get_text(result))

I'm sorry, but I don't know who "she" is. Could you please tell me her name?


## Key Takeaways

1. Without a checkpointer, every `invoke()` is a **blank slate** — no conversation history
2. `InMemorySaver()` + `thread_id` gives the agent **short-term memory** across turns
3. Different `thread_id`s keep conversations **isolated** — like separate chat sessions